# Reproducing Gable et al. (2026), Figure S4 and Table 5, from DANDI:000010

**Paper.** Gable J, Newman ZL, Young S, Scheib J, Bliese S, Miller N, Kerlin A (2026). Tuft dendrite
spikes are accompanied by selective input from distinct functional networks. *J Neurosci* 46(38)
e0974262026, [doi:10.1523/jneurosci.0974-26.2026](https://doi.org/10.1523/jneurosci.0974-26.2026).
The claims file (`claims/gable-2026.yaml`) was written from the version of record (PMC13612898)
with its supplement, and from the paper's code, github.com/kerlin-lab/Gable_2026 at commit 688e662.

**Dataset.** [DANDI:000010](https://dandiarchive.org/dandiset/000010/0.220126.1905), version
0.220126.1905, "A motor cortex circuit for motor planning and movement" (Li, Chen, Guo, Gerfen and
Svoboda 2015). The paper reuses its silicon-probe recordings of ALM for Figure S4 and Table 5 only.
It cites the dataset without a version. 0.220126.1905 is the only published version; the draft
holds the same 158 assets (same paths, asset ids and sizes, checked below) and differs only in
dandiset metadata, so the published version is pinned.

**What can be tested.** The four claims on the silicon-probe recordings (figs4-*). Every other claim
rests on the paper's own two-photon imaging, which is not in this dataset nor in any public archive
the paper or its code names. The claims file marks them `data: dataset`, and they are blocked as
`data_missing` (a reader's answer in `questions/gable-2026.yaml` changed them from `data: elsewhere`).

| Claim | Tested here |
|---|---|
| figs4-session-selection | yes |
| figs4-lowest-couplers-correlated | yes |
| figs4-high-couplers-correlated | yes |
| figs4-middle-couplers-not-correlated | yes |
| 30 imaging claims (fig1 to fig5, figs1 to figs3, table1) | no, blocked (`data_missing`): the imaging data are not public |

**Method.** The claims file's E1 to E8 (selection, binning, trial-specific activity, population
coupling, pairs, bootstrap test) are implemented in `alm_coupling.py`, whose docstring says where
each parameter comes from. The open details O1 to O3 are run under every reading:

- O1 (a) all high-quality units of a selected session, (b) only those above 0.5 Hz;
- O2 (a) rate over the time each unit was observed, (b) rate within the analysis windows;
- O3 (a) only the four trial types CL, CR, IL, IR, (b) all trials with sample and GO times.

Each combination is written as three letters, `aaa` to `bbb`. Units stored twice in a file (see the
data check) are counted once; the counts with every stored row, as a variant, are shown for the
selection claim.

In [ ]:
import resource
import time
import warnings
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

import alm_coupling as ac

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

PAPER = "gable-2026"
started = time.time()
assets = ac.ecephys_assets()
sessions = ac.load_all(assets)
url_of = dict(assets)
print(f"loaded {len(sessions)} ecephys files of dandiset {ac.DANDISET_ID} version {ac.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
api = "https://api.dandiarchive.org/api/dandisets/000010/versions/{}/assets/?page_size=500"
listing = {v: {(a["path"], a["asset_id"], a["size"]) for a in requests.get(api.format(v)).json()["results"]}
           for v in [ac.DANDISET_VERSION, "draft"]}
print("assets, published:", len(listing[ac.DANDISET_VERSION]), " draft:", len(listing["draft"]),
      " differing (path, asset id, size):", len(listing[ac.DANDISET_VERSION] ^ listing["draft"]))

info = pd.DataFrame([{
    "subject": s["subject"], "session": s["session"], "genotype": s["genotype"], "strains": s["strains"],
    "group": ac.strain_group(s["strains"]), "sex": s["sex"],
    "trials": len(s["trials"]), "task": "/".join(sorted(s["trials"].task.unique())),
    "units": len(s["units"]), "unique_units": len(ac.unique_units(s)),
    "quality": "/".join(sorted(s["units"].quality.unique())),
    "photostim_events": int(s["trials"].photostim_start_time.notna().sum()),
    "events_nan_time": int(sum(s["unmatched_events"].values())),
} for s in sessions])
print("\nFiles:", len(info), " subjects:", info.subject.nunique())
print("\nMice and sessions per genotype (subject description):")
print(info.groupby(["group", "genotype", "strains"], dropna=False).agg(
    mice=("subject", "nunique"), sessions=("subject", "size"), units=("units", "sum")).to_string())
print("\nQuality labels over all units:", info.quality.unique().tolist())
print("Task label of the trials:", info.task.unique().tolist())
print("Cell types:", pd.Series(np.concatenate([s["units"].cell_type for s in sessions])).value_counts().to_dict())
print("Units per file: min", info.units.min(), "median", info.units.median(), "max", info.units.max())
print("\n", info.drop(columns=["strains"]).to_string(index=False))

The subject records give five genotypes. KJ18 is the Sim1-Cre line and PL56 the Tlx3-Cre line
named by the paper; ten mice carry KJ18 (one of them crossed with Ai32) and six PL56 (two crossed
with Ai32); three Rbp4-Cre (KL100) mice are not among the paper's genotypes and are left out. Every
unit is labelled "good", so the paper's "high-quality units, as labeled in the original dataset"
keeps every unit. Every trial is labelled `audio delay`, while the dataset's experiment description
("object location discrimination task") and the paper (tactile sample cue) describe a tactile
task: the task label disagrees with the description.

In [ ]:
dup_rows, dup_identical, files_with_dups = 0, 0, 0
for s in sessions:
    ids = s["units"].id.to_numpy()
    vals, counts = np.unique(ids, return_counts=True)
    if (counts > 1).any():
        files_with_dups += 1
    for v in vals[counts > 1]:
        w = np.where(ids == v)[0]
        dup_rows += len(w) - 1
        dup_identical += all(np.array_equal(s["spikes"][w[0]], s["spikes"][j]) for j in w[1:])
print(f"files with a repeated unit id: {files_with_dups} of {len(sessions)}; extra rows: {dup_rows}; "
      f"repeated ids whose rows hold identical spike trains: {dup_identical}")
print("units in all files:", int(info.units.sum()), " counted once:", int(info.unique_units.sum()))
example = info[info.units > info.unique_units].iloc[0]
s = next(x for x in sessions if x["subject"] == example.subject and x["session"] == example.session)
print("example", s["path"], "unit ids:", s["units"].id.tolist())

# identical spike trains across files (a recording deposited twice)
keys = {}
for s in sessions:
    for j in ac.unique_units(s):
        keys.setdefault((len(s["spikes"][j]), s["spikes"][j][:20].tobytes()), set()).add(s["path"])
print("spike trains found in more than one file:", sum(len(v) > 1 for v in keys.values()))
starts = {}
for s in sessions:
    starts.setdefault(np.round(s["trials"].start.to_numpy(), 3).tobytes(), []).append(s["path"])
print("files sharing an identical list of trial start times:", [v for v in starts.values() if len(v) > 1])

# spike times against the observation intervals
rows = []
for s in sessions:
    ob0 = s["obs"][0]
    tr = s["trials"]
    has = tr["sample"].notna().to_numpy()
    out = 0
    for sp, ob in zip(s["spikes"], s["obs"]):
        i = np.searchsorted(ob[:, 0], sp, side="right") - 1
        out += int(np.sum(~((i >= 0) & (sp <= ob[np.clip(i, 0, None), 1]))))
    rows.append({"path": s["path"], "units_share_intervals": all(np.array_equal(o, ob0) for o in s["obs"]),
                 "intervals": len(ob0), "trials_with_sample": int(has.sum()),
                 "trials_with_interval": int(np.isfinite(ac.observed_until(s)).sum()),
                 "sample_but_no_interval": int((has & ~np.isfinite(ac.observed_until(s))).sum()),
                 "interval_lengths": "/".join(map(str, np.unique(np.round(ob0[:, 1] - ob0[:, 0], 3)))),
                 "stop_minus_interval_end_max": float(np.nanmax(np.abs(tr.stop.to_numpy() - ac.observed_until(s)))),
                 "spikes_outside": out, "unsorted_units": s["unsorted_units"]})
obs_check = pd.DataFrame(rows)
print("files whose units share one set of observation intervals:", int(obs_check.units_share_intervals.sum()), "of", len(obs_check))
print("intervals:", int(obs_check.intervals.sum()), " intervals starting at a trial start:", int(obs_check.trials_with_interval.sum()),
      " trials with a sample time:", int(obs_check.trials_with_sample.sum()),
      " of these without an interval:", int(obs_check.sample_but_no_interval.sum()),
      "in", int((obs_check.sample_but_no_interval > 0).sum()), "files")
print("interval lengths (s):", obs_check.interval_lengths.unique().tolist())
print("largest difference between a trial's stop time and its interval's end (s):",
      round(obs_check.stop_minus_interval_end_max.max(), 3))
print("spikes outside their unit's observation intervals, by file:")
print(obs_check[obs_check.spikes_outside > 0][["path", "spikes_outside"]].to_string(index=False))
print("units whose spike times are stored out of order (sorted on loading), by file:")
print(obs_check[obs_check.unsorted_units > 0][["path", "unsorted_units"]].to_string(index=False))

In 50 of the 99 files some unit ids appear twice; in every case the two rows hold the same spike
train (for example ids 3, 4, 4, 5, 5), so the file stores one unit twice. Whether another unit
was lost in its place cannot be told from the files. Units are counted once here. No spike train or list of trial
starts is repeated across files. Every unit of a file shares one set of observation intervals,
one per trial with a sample time, starting at the trial start and lasting 5.5 s; the trials'
stop times differ from the interval ends. Spikes are stored only within these intervals (apart
from the files listed), so a unit's "session-wide" rate is its rate over those intervals. A few
units store spike times out of order; they are sorted on loading.

In [ ]:
allt = pd.concat([s["trials"].assign(path=s["path"]) for s in sessions], ignore_index=True)
print("trials:", len(allt))
print("\noutcome by early lick:")
print(pd.crosstab(allt.early_lick, allt.outcome).to_string())
print("\ntrials without a sample time, by outcome:")
print(allt[allt["sample"].isna()].outcome.value_counts().to_string())
d_sd = (allt.delay - allt["sample"]).dropna()
d_dg = (allt.go - allt.delay).dropna()
print(f"\nsample to delay onset (s): median {d_sd.median():.4f}, within 10 ms of 1.3 s: {np.mean(abs(d_sd - 1.3) < 0.01):.3f}")
print(f"delay to GO (s): median {d_dg.median():.4f}, within 10 ms of 1.3 s: {np.mean(abs(d_dg - 1.3) < 0.01):.3f}")
print(f"trial length (stop - start, s): median {(allt.stop - allt.start).median():.2f}, "
      f"range {(allt.stop - allt.start).min():.2f} to {(allt.stop - allt.start).max():.2f}")
print(f"sample onset after trial start (s): median {(allt['sample'] - allt.start).median():.3f}")
ph = allt.dropna(subset=["photostim_start_time"])
print(f"\nphotostimulation onset relative to sample onset (s): median {(ph.photostim_start_time - ph['sample']).median():.3f}, "
      f"range {(ph.photostim_start_time - ph['sample']).min():.3f} to {(ph.photostim_start_time - ph['sample']).max():.3f}")
print("events with a NaN timestamp, by event:",
      pd.DataFrame([s["unmatched_events"] for s in sessions]).sum().astype(int).to_dict())

The sample and delay epochs are each 1.3 s in most trials (Li et al. 2015), so the window from
sample onset to 2 s after GO is 4.6 s; an early lick restarts the epoch and lengthens the window.
Some events in the BehavioralEvents series have a NaN timestamp; they cannot be placed in a trial.
Whether a trial was photostimulated is read from the photostimulation onsets; it is checked below
against the stored laser trace for the selected sessions.

### Clock check

The spikes and the trial events must share a clock. ALM activity predicts the instructed lick
direction after the sample cue and most strongly after GO (Li et al. 2015), and should not predict
it before the sample cue. The instructed side of correct trials is decoded (nearest class mean,
five interleaved folds, balanced accuracy) from each unit's spike count in windows relative to
sample onset, in every selected session of reading `baa` (adopted below); as a control each trial takes the next
trial's label.

In [ ]:
main = ("b", "a", "a")   # the reading adopted below, under "Population coupling"
W = [(-0.5, 0.0), (0.0, 0.5), (0.8, 1.3), (1.8, 2.6), (2.6, 3.1), (3.1, 3.6), (3.6, 4.6)]
wnames = ["-0.5-0 (pre-sample)", "0-0.5 (sample)", "0.8-1.3", "1.8-2.6 (delay)", "2.6-3.1 (after GO)", "3.1-3.6", "3.6-4.6"]
rows, rows_shift = [], []
for s in sessions:
    if ac.strain_group(s["strains"]) is None:
        continue
    sel = ac.select_session(s, *main)
    if not sel["passes"]:
        continue
    rows.append(ac.decode_side(s, sel["units"], sel["trial_mask"], "sample", W))
    rows_shift.append(ac.decode_side(s, sel["units"], sel["trial_mask"], "sample", W, shift_labels=True))
clock = pd.DataFrame({"window from sample onset (s)": wnames,
                      "median accuracy": np.median(rows, 0), "min": np.min(rows, 0), "max": np.max(rows, 0),
                      "median, labels shifted one trial": np.median(rows_shift, 0)})
print(f"sessions: {len(rows)}")
print(clock.round(3).to_string(index=False))

The accuracy printed before sample onset and with shifted labels is the chance reference; the
accuracies after sample onset and after GO are compared with it in the cell's output. A rise only
after the cue, and the loss of that rise with shifted labels, is consistent with spikes and trial
events on one clock.

In [ ]:
# photostimulated trials: onsets in BehavioralEvents against the stored laser power trace
rows = []
for s in sessions:
    if ac.strain_group(s["strains"]) is None or "ogen" not in s["path"]:
        continue
    sel = ac.select_session(s, *main)
    if not sel["passes"]:
        continue
    mx, names = ac.laser_max_per_trial(url_of[s["path"]], s["trials"])
    ph = s["trials"].photostim_start_time.notna().to_numpy()
    rows.append({"session": s["path"].split("/")[1][:30], "laser": "/".join(names),
                 "onset_trials": int(ph.sum()), "onset_trials_with_power>1": int((mx[ph] > 1).sum()),
                 "other_trials_with_power>1": int((mx[~ph] > 1).sum()),
                 "included_trials_with_power>1": int((mx[sel["trial_mask"]] > 1).sum()),
                 "max_power": float(mx.max())})
laser = pd.DataFrame(rows)
print(laser.to_string(index=False))

## figs4-session-selection

> The reanalysis of the Li et al. (2015) ALM silicon-probe recordings includes eight Sim1_KJ18-Cre
> and three Tlx_PL56-Cre mice, 20 sessions, and yields 753 same-coupling-range and 6,586
> different-coupling-range neuron pairs.

Criterion: under one reading of O1 to O3, mice exactly 11 (8 Sim1, 3 Tlx), sessions 20 ± 2, same
753 ± 75, different 6,586 ± 659 (summed over the 7 bins). Mice are counted by the strain of the
subject record; the KJ18 and PL56 mice crossed with Ai32 are counted with their Cre line, and the
count without them is shown too.

In [ ]:
readings = [(a, b, c) for a in "ab" for b in "ab" for c in "ab"]
results, results_dup = {}, {}
for r in readings:
    results[r] = ac.analyse(sessions, *r, dedupe=True)
    results_dup[r] = ac.analyse(sessions, *r, dedupe=False)

def counts(res):
    ss = res["sessions"]
    mice = ss.drop_duplicates("subject")
    return {"mice": len(mice), "Sim1": int((mice.group == "Sim1").sum()), "Tlx": int((mice.group == "Tlx").sum()),
            "mice_without_Ai32": int((~mice.ai32).sum()), "sessions": len(ss),
            "units_min": int(ss.n_units.min()), "units_max": int(ss.n_units.max()),
            "same": int(sum(sum(len(v) for v in b) for b in res["same"])),
            "different": int(sum(sum(len(v) for v in b) for b in res["diff"]))}

sel_table = pd.DataFrame([{"reading": "".join(r), "units": "counted once", **counts(results[r])} for r in readings] +
                         [{"reading": "".join(r), "units": "every stored row", **counts(results_dup[r])} for r in readings])
sel_table["match"] = ((sel_table.mice == 11) & (sel_table.Sim1 == 8) & (sel_table.Tlx == 3)
                      & (abs(sel_table.sessions - 20) <= 2) & (abs(sel_table.same - 753) <= 75)
                      & (abs(sel_table.different - 6586) <= 659))
print("paper: 11 mice (8 Sim1, 3 Tlx), 20 sessions, 753 same and 6,586 different pairs; "
      "the code's printed output: 20 sessions of 18 to 25 neurons, 771 same-range pairs")
print(sel_table.to_string(index=False))
print("\nreadings matching all four values:", sel_table[sel_table.match].reading.tolist())

In [ ]:
r = main
print("reading baa, units counted once; sessions kept:")
print(results[r]["sessions"][["subject", "session", "group", "ai32", "n_units", "n_trials"]].to_string(index=False))
per_mouse = pd.concat([results[r]["sessions"].assign(reading="".join(r)) for r in readings])
print("\nsessions per mouse under each reading (units counted once):")
print(pd.crosstab(per_mouse.subject + " " + per_mouse.group, per_mouse.reading).to_string())
print("\nKJ18 and PL56 mice in the dataset with no selected session under any reading:",
      sorted(set(info[info.group.notna()].subject) - set(per_mouse.subject)))

**Result: not reproduced.** No reading of O1 to O3 selects the paper's 11 mice. With units counted
once every reading gives 9 mice (7 Sim1, 2 Tlx) and 15 or 16 sessions; with every stored row, 10
mice (7 Sim1, 3 Tlx) and 17 or 18 sessions, against 8 Sim1 and 3 Tlx mice and 20 sessions. Under
reading `baa`, adopted below, there are 501 same-range and 5,060 different-range pairs (paper 753
and 6,586; the code printed 771 same-range pairs); over all readings the counts run from 452 to
893 and from 4,650 to 8,860. Seven of the 16 KJ18 and PL56 mice have no session that passes the
filter under any reading.

## Population coupling and same against different coupling range

The three remaining claims share one computation (E4 to E8) and differ in the bins they judge. No
reading matched the selection claim, so, as the claims file directs, every reading is run. Where
the readings give different outcomes, the outcome is taken under reading `baa`, adopted as an
assumption: O1 (b), only the units above 0.5 Hz, because the output printed by the paper's
FigureS4.ipynb (commit 688e662) gives 18 to 25 neurons per session, which readings `baa` and `bab`
give here (table above) and the O1 (a) readings do not (up to 34); O2 (a), the rate over the
recording, as "session-wide" says; O3 (a), the four trial types, as TSA is defined on CL, CR, IR
and IL (Methods, Activity extraction). The results under every reading are shown. Each
claim takes its own bootstrap generator, seeded from the claim, and is repeated under the four
further seeds (claim seed plus 1000 to 4000). The distribution of population coupling and the
seven bin edges are printed first.

In [ ]:
for r in readings:
    c = np.concatenate(results[r]["coupling"])
    print("".join(r), f"neurons {len(c)}, coupling median {np.median(c):.3f}, range {c.min():.3f} to {c.max():.3f}, "
          f"edges {np.round(results[r]['edges'], 3).tolist()}")
c = np.concatenate(results[main]["coupling"])
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(c, bins=40, color="0.3")
for e in results[main]["edges"]:
    ax.axvline(e, color="C1", lw=0.8)
ax.set_xlabel("population coupling (r), reading baa")
ax.set_ylabel("neurons")
plt.tight_layout()
plt.show()
print("all values are inside the axes; range", round(c.min(), 3), "to", round(c.max(), 3))

In [ ]:
def run_claim(claim_id):
    seed = zlib.crc32(f"{PAPER}/{claim_id}".encode())
    out = {}
    for r in readings:
        for extra in [0, 1000, 2000, 3000, 4000]:
            rng = np.random.default_rng(seed + extra)
            out[(r, extra)] = ac.same_vs_diff_test(results[r]["same"], results[r]["diff"], rng)
    return seed, out

BIN_NAMES = ["0.00-14.29", "14.29-28.57", "28.57-42.86", "42.86-57.14", "57.14-71.43", "71.43-85.71", "85.71-100.00"]
PAPER_MD = [0.06, 0.004, 0.001, -0.002, 0.02, 0.06, 0.1]
PAPER_P = [7e-4, 0.12, 0.61, 0.76, 7e-4, 7e-4, 7e-4]

def table(out, extra=0):
    rows = []
    for r in readings:
        t = out[(r, extra)]
        for k in range(7):
            rows.append({"reading": "".join(r), "bin": BIN_NAMES[k], "n_same": t.n_same[k], "n_diff": t.n_diff[k],
                         "same": t.same_mean[k], "different": t.diff_mean[k],
                         "mean_difference": t.mean_difference[k], "p_holm": t.p_holm[k]})
    return pd.DataFrame(rows)

def seed_range(out, bins):
    rows = []
    for r in readings:
        for k in bins:
            md_ = [out[(r, e)].mean_difference[k] for e in [0, 1000, 2000, 3000, 4000]]
            p_ = [out[(r, e)].p_holm[k] for e in [0, 1000, 2000, 3000, 4000]]
            rows.append({"reading": "".join(r), "bin": BIN_NAMES[k], "md_min": min(md_), "md_max": max(md_),
                         "p_holm_min": min(p_), "p_holm_max": max(p_)})
    return pd.DataFrame(rows)

## figs4-lowest-couplers-correlated

> In the ALM silicon-probe recordings, neurons in the lowest population-coupling bin (0 to 14.29%)
> are more correlated with each other than with neurons of other coupling bins, as the synapses with
> the lowest coupling are.

Criterion: reproduced when the bin 1 mean difference is positive with Holm-corrected one-tailed
p < 0.05 and within 0.06 ± 0.006; partially reproduced when positive and significant but outside
tolerance; not reproduced when the Holm-corrected p is 0.05 or more.

In [ ]:
seed_low, out_low = run_claim("figs4-lowest-couplers-correlated")
t = table(out_low)
low = t[t.bin == BIN_NAMES[0]].copy()
low["outcome"] = np.where(low.p_holm >= 0.05, "not reproduced",
                          np.where((low.mean_difference > 0) & (abs(low.mean_difference - 0.06) <= 0.006), "reproduced",
                                   np.where(low.mean_difference > 0, "partially reproduced", "not reproduced")))
print(f"claim seed {seed_low}; paper: mean difference 0.06, Holm p 7.0e-4")
print(low.round(4).to_string(index=False))
print("\nover the five seeds:")
print(seed_range(out_low, [0]).round(4).to_string(index=False))
print("\nall seven bins, reading baa, claim seed:")
print(out_low[(main, 0)].assign(bin=BIN_NAMES, paper_md=PAPER_MD, paper_p=PAPER_P).round(4).to_string(index=False))

In [ ]:
bm = ac.by_mouse_differences(results[main])
print("reading baa: per mouse, mean same minus mean different correlation per bin (no test; n_same is the same-range pair count)")
print(bm.round(3).to_string(index=False))
print("\nmice with a positive bin 1 difference:", int((bm.bin1 > 0).sum()), "of", int(bm.bin1.notna().sum()))
fig, ax = plt.subplots(figsize=(6, 3))
tt = out_low[(main, 0)]
ax.plot(range(1, 8), tt.same_mean, "ko", label="same range")
ax.plot(range(1, 8), tt.diff_mean, "^", color="0.5", label="different range")
ax.set_xticks(range(1, 8))
ax.set_xlabel("coupling bin (1 = lowest)")
ax.set_ylabel("mean pair correlation")
ax.legend()
plt.tight_layout()
plt.show()
print("plotted values range", round(float(min(tt.same_mean.min(), tt.diff_mean.min())), 4), "to",
      round(float(max(tt.same_mean.max(), tt.diff_mean.max())), 4), "; all inside the axes")

**Result: partially reproduced, under the assumed reading `baa`.** The lowest bin's mean
difference is 0.048 (same 0.028, different -0.020; Holm p = 0.0007), positive and significant but
below the tolerance of 0.054 to 0.066. It is significant under all eight readings, from 0.040 to
0.055, and within tolerance only under `bba`; over the five seeds it is 0.0478 to 0.0479 under
`baa`. The difference is positive in each of the 8 mice with same-range pairs in the bin.

## figs4-high-couplers-correlated

> In the ALM silicon-probe recordings, neurons in the three highest coupling bins (57.14 to 100%)
> are more correlated with neurons of the same bin than with neurons of other bins (Table 5
> differences 0.02, 0.06 and 0.1), as for the synapses.

Criterion: reproduced when all three bins have a positive mean difference with Holm-corrected
one-tailed p < 0.05 and within tolerance (0.015 to 0.025, 0.054 to 0.066, 0.05 to 0.15);
partially reproduced when all three are significant and one or more is outside tolerance, or two of
three are significant and within tolerance.

In [ ]:
seed_high, out_high = run_claim("figs4-high-couplers-correlated")
t = table(out_high)
lo_hi = {4: (0.015, 0.025), 5: (0.054, 0.066), 6: (0.05, 0.15)}
rows = []
for r in readings:
    tt = out_high[(r, 0)]
    sig = [(tt.mean_difference[k] > 0) and (tt.p_holm[k] < 0.05) for k in (4, 5, 6)]
    tol = [lo_hi[k][0] <= tt.mean_difference[k] <= lo_hi[k][1] for k in (4, 5, 6)]
    if all(sig) and all(tol):
        o = "reproduced"
    elif all(sig) or sum(s and w for s, w in zip(sig, tol)) >= 2:
        o = "partially reproduced"
    else:
        o = "not reproduced"
    rows.append({"reading": "".join(r), **{f"md_{BIN_NAMES[k]}": tt.mean_difference[k] for k in (4, 5, 6)},
                 **{f"p_{BIN_NAMES[k]}": tt.p_holm[k] for k in (4, 5, 6)},
                 "significant": sum(sig), "within_tolerance": sum(tol), "outcome": o})
high = pd.DataFrame(rows)
print(f"claim seed {seed_high}; paper: 0.02, 0.06, 0.1, each Holm p 7.0e-4")
print(high.round(4).to_string(index=False))
print("\nover the five seeds:")
print(seed_range(out_high, [4, 5, 6]).round(4).to_string(index=False))
print("\npair counts, reading baa:")
print(t[(t.reading == "baa")][["bin", "n_same", "n_diff"]].to_string(index=False))

In [ ]:
print("reading baa, per mouse, bins 5 to 7 (no test):")
print(bm[["subject", "sessions", "bin5", "bin6", "bin7", "n_same5", "n_same6", "n_same7"]].round(3).to_string(index=False))
for k in (5, 6, 7):
    print(f"bin {k}: mice with a positive difference {int((bm[f'bin{k}'] > 0).sum())} of {int(bm[f'bin{k}'].notna().sum())}")

**Result: not reproduced, under the assumed reading `baa`.** Bins 71.43 to 85.71% and 85.71 to
100% are significant (mean differences 0.021 and 0.067, Holm p = 0.0007), but bin 57.14 to 71.43%
is not (0.008, Holm p = 0.074; 0.066 to 0.090 over the five seeds), and only the top bin is within
tolerance (the middle one is about a third of the paper's 0.06). Under `bab` all three bins are
significant, which the criterion calls partially reproduced; under the other six readings the
57.14 to 71.43% bin is not significant.

## figs4-middle-couplers-not-correlated

> In the ALM silicon-probe recordings, the three middle coupling bins (14.29 to 57.14%) show no
> significant excess of same-bin over different-bin correlation.

Criterion: a claim of absence on the paper's test (E8, one-tailed, Holm across the 7 bins,
alpha 0.05). Reproduced when none of bins 2, 3 and 4 has a Holm-corrected one-tailed p below
0.05; not reproduced when any does. The three mean differences are reported either way.

In [ ]:
seed_mid, out_mid = run_claim("figs4-middle-couplers-not-correlated")
rows = []
for r in readings:
    tt = out_mid[(r, 0)]
    rows.append({"reading": "".join(r), **{f"md_{BIN_NAMES[k]}": tt.mean_difference[k] for k in (1, 2, 3)},
                 **{f"p_{BIN_NAMES[k]}": tt.p_holm[k] for k in (1, 2, 3)},
                 "outcome": "reproduced" if all(tt.p_holm[k] >= 0.05 for k in (1, 2, 3)) else "not reproduced"})
mid = pd.DataFrame(rows)
print(f"claim seed {seed_mid}; paper: 0.004 (p 0.12), 0.001 (p 0.61), -0.002 (p 0.76)")
print(mid.round(4).to_string(index=False))
print("\nover the five seeds:")
print(seed_range(out_mid, [1, 2, 3]).round(4).to_string(index=False))
print("\nreading baa, per mouse (no test):")
print(bm[["subject", "bin2", "bin3", "bin4"]].round(3).to_string(index=False))

**Result: reproduced, under the assumed reading `baa`.** None of the three middle bins has a
Holm-corrected p below 0.05: mean differences -0.001, -0.003 and 0.005, Holm p 1.0, 1.0 and 0.54.
Under four of the eight readings (`aba`, `abb`, `bab`, `bbb`) one of these bins does reach
p < 0.05 (smallest 0.007, under `abb`), with mean differences of at most 0.0084.

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| figs4-session-selection | not reproduced (9 or 10 mice and 15 to 18 sessions, against 11 and 20) |
| figs4-lowest-couplers-correlated | partially reproduced, assumed reading (0.048 against 0.06, significant) |
| figs4-high-couplers-correlated | not reproduced, assumed reading (57.14 to 71.43% bin not significant) |
| figs4-middle-couplers-not-correlated | reproduced, assumed reading |
| 30 claims on the paper's imaging | blocked (`data_missing`), the data are not public |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The step that builds the paper's per-session arrays from
these files is not in its code, so the unit filter (O1), the rate (O2) and the trial set (O3) are
run under every reading and the reading `baa` is adopted where outcomes differ; the window is a
fixed 4.6 s from sample onset (O5); units stored twice are counted once. The selection does not
reach the paper's 11 mice and 20 sessions under any reading, so the population differs from the
paper's, which bears on the three test claims.

What was found about the dataset. Every unit is labelled "good"; 50 of 99 files store 72 units
twice under a repeated id; every trial is labelled `audio delay` though the task described is
tactile; some event timestamps are NaN; a few trials with sample times have no observation
interval; a few units store spike times out of order, and two files hold spikes outside the
observation intervals. The spikes and trial events share a clock as far as the decoding check
shows, and photostimulation onsets agree with the stored laser trace in the selected sessions.